# AtomicVision GRPO Colab Bridge

This notebook is the Phase 10 bridge and Phase 11 launch path for fine-tuning. It verifies the deployed OpenEnv Space, runs a simple environment action loop, then shows the minimal TRL GRPO commands for Colab or Kaggle.

Deployed Space: https://prodigyhuh-atomicvision-openenv.hf.space

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = "https://huggingface.co/spaces/prodigyhuh/atomicvision-openenv"
PROJECT_DIR = "AtomicVision"

if not pathlib.Path("training/train_grpo_atomicvision.py").exists():
    if not pathlib.Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

!pip install -q requests websockets
!pip install -q -r training/requirements-grpo.txt

In [ ]:
import json
import requests

SPACE_BASE = "https://prodigyhuh-atomicvision-openenv.hf.space"

health = requests.get(f"{SPACE_BASE}/health", timeout=60)
print(health.status_code, health.text)
health.raise_for_status()

In [ ]:
reset = requests.post(f"{SPACE_BASE}/reset", json={"seed": 42}, timeout=60)
reset.raise_for_status()
observation = reset.json()["observation"]
print(json.dumps({
    "material_id": observation["material_id"],
    "difficulty": observation["difficulty"],
    "candidate_defects": observation["candidate_defects"],
    "budget_remaining": observation["budget_remaining"],
}, indent=2))

In [ ]:
import websockets

WS_URL = SPACE_BASE.replace("https://", "wss://") + "/ws"

async def run_prior_submit_episode(seed=42):
    async with websockets.connect(WS_URL, open_timeout=60) as ws:
        await ws.send(json.dumps({"type": "reset", "data": {"seed": seed}}))
        reset_msg = json.loads(await ws.recv())
        first_obs = reset_msg["data"]["observation"]

        await ws.send(json.dumps({"type": "step", "data": {"action_type": "ask_prior"}}))
        prior_msg = json.loads(await ws.recv())
        prior_obs = prior_msg["data"]["observation"]
        prior = prior_obs["prior_prediction"]

        submit_action = {
            "type": "step",
            "data": {
                "action_type": "submit_defect_map",
                "predicted_defects": prior["predicted_defects"],
                "predicted_concentrations": prior["predicted_concentrations"],
                "confidence": prior["confidence"],
            },
        }
        await ws.send(json.dumps(submit_action))
        submit_msg = json.loads(await ws.recv())
        await ws.send(json.dumps({"type": "close", "data": {}}))
        submit_payload = submit_msg["data"]
        submit_obs = submit_payload["observation"]
        submit_obs["reward"] = submit_payload["reward"]
        submit_obs["done"] = submit_payload["done"]
        return first_obs, prior_obs, submit_obs

first_obs, prior_obs, submit_obs = await run_prior_submit_episode(seed=42)
print("episode:", submit_obs["episode_id"])
print("prior:", json.dumps(prior_obs["prior_prediction"], indent=2))
print("reward:", submit_obs["reward"])
print(json.dumps(submit_obs["reward_breakdown"], indent=2))

## Phase 11 GRPO Fine-Tuning

Fine-tuning starts here. The first goal is a smoke run that proves the model can call AtomicVision tools, receive final rewards, and produce logs. The target for the judged run is to beat the `prior_submit` baseline in `docs/reward-comparison-report.md`.

Recommended order:

1. Run dry-run first. It does not download a base LLM.
2. Run a 5-step LoRA smoke training.
3. If stable, increase to 100-300 steps for the demo reward curve.
4. Push only the final adapter run to Hugging Face Hub.

In [ ]:
!python training/train_grpo_atomicvision.py --dry-run --difficulty easy

Run this only after the dry-run succeeds and the runtime has a GPU attached.

In [ ]:
!python training/train_grpo_atomicvision.py --preset smoke --report-to trackio

After this reaches `100% 5/5`, run the 20-step learning pass below.

In [ ]:
!python training/train_grpo_atomicvision.py --preset colab-20 --report-to trackio

For more parameters, move to `Qwen/Qwen3-1.7B` after the 0.6B run is stable. Add `--push-to-hub --hub-model-id prodigyhuh/atomicvision-qwen3-1p7b-grpo-lora` only after logging in with a Hugging Face write token.

In [ ]:
!python training/train_grpo_atomicvision.py --preset qwen-1p7b-50 --report-to trackio